# 🎓 Modelo Predictivo de Deserción Escolar y Sistema de Alerta Temprana (CRISP-ML)

**Maestría en Inteligencia Artificial Aplicada a la Educación e Investigación**  
**Asignatura:** Fundamentos de IA y Aprendizaje Automático  
**Proyecto Final Integrador**  
**Autora:** Yudelka Morel  
**Profesor:** Dr. Darwin Muñoz, Ph.D  
**Fecha:** Junio 2026

---

## 1️⃣ Fase 1: Comprensión del Negocio (Business Understanding)

### Contexto y Reto
La deserción escolar en la República Dominicana representa una problemática apremiante con consecuencias a largo plazo para el desarrollo nacional. Según datos del Ministerio de Educación (MINERD), las tasas de abandono superan el 20% en secundaria. A menudo, las intervenciones llegan demasiado tarde. Este proyecto busca predecir de forma temprana el riesgo de deserción mediante Aprendizaje Automático para posibilitar intervenciones oportunas.

### Objetivos del Proyecto
* **Objetivo General:** Desarrollar y documentar un modelo de Machine Learning que identifique factores de riesgo de deserción y alimente un sistema de alerta temprana.
* **Objetivos Específicos:**
  1. Identificar y evaluar variables académicas, socioeconómicas y conductuales más incidentes.
  2. Construir un modelo clasificador (**Regresión Logística** y **Random Forest**) para predecir si un estudiante abandonará el año escolar.
  3. Construir un modelo regresor (**Regresión Lineal**) para estimar el rendimiento académico (`promedio_calificaciones`) del estudiante.
  4. Diseñar un prototipo del Sistema de Alerta Temprana (SAT) con tres niveles de riesgo (Bajo/Medio/Alto) y sus respectivos protocolos pedagógicos.
  5. Auditar sesgos del modelo en subgrupos vulnerables (género y zona rural/urbana).


## 2️⃣ Fase 2: Comprensión de los Datos (Data Understanding)

### Configuración del entorno e importación de librerías


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, cohen_kappa_score,
                              confusion_matrix, classification_report, roc_curve, precision_recall_curve,
                              mean_squared_error, r2_score)
from imblearn.over_sampling import SMOTE
import joblib

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Librerías cargadas correctamente.')


### Generación del Dataset
Generamos un conjunto de datos simulado de 4,500 estudiantes de educación secundaria que combina variables académicas, socioeconómicas y conductuales representativas de la realidad de República Dominicana.


In [ ]:
def generar_dataset(n=4500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    edad = rng.integers(14, 21, n)
    genero = rng.choice(['M', 'F'], n)
    zona = rng.choice(['Urbana', 'Rural'], n, p=[0.65, 0.35])
    nivel_socioeconomico = rng.choice(['Bajo', 'Medio', 'Alto'], n, p=[0.35, 0.45, 0.20])
    educacion_padre = rng.choice(['Ninguno', 'Primaria', 'Secundaria', 'Superior'], n, p=[0.15, 0.35, 0.35, 0.15])
    educacion_madre = rng.choice(['Ninguno', 'Primaria', 'Secundaria', 'Superior'], n, p=[0.12, 0.33, 0.37, 0.18])
    apoyo_familiar = rng.choice(['Bajo', 'Medio', 'Alto'], n, p=[0.25, 0.45, 0.30])
    
    vulnerabilidad = (
        (nivel_socioeconomico == 'Bajo').astype(float) * 0.3 +
        (apoyo_familiar == 'Bajo').astype(float) * 0.3 +
        (zona == 'Rural').astype(float) * 0.2 +
        rng.normal(0, 0.2, n)
    )
    
    tasa_asistencia = np.clip(95 - vulnerabilidad * 35 + rng.normal(0, 6, n), 30, 100)
    promedio_calificaciones = np.clip(80 - vulnerabilidad * 25 + rng.normal(0, 8, n), 30, 100)
    reprobaciones_previas = np.clip((vulnerabilidad * 4 + rng.poisson(0.6, n)).astype(int), 0, 8)
    incidencias_disciplinarias = np.clip((vulnerabilidad * 3 + rng.poisson(0.4, n)).astype(int), 0, 15)
    cambios_escuela = np.clip((vulnerabilidad * 1.5 + rng.poisson(0.2, n)).astype(int), 0, 5)
    distancia_escuela = np.clip(rng.exponential(3, n) + (zona == 'Rural').astype(float) * 5, 0, 50)
    
    trabaja_estudiante = (rng.random(n) < (0.10 + vulnerabilidad * 0.25)).astype(int)
    acceso_internet = (rng.random(n) > (0.15 + vulnerabilidad * 0.35)).astype(int)
    actividades_extracurriculares = (rng.random(n) > (0.4 + vulnerabilidad * 0.2)).astype(int)
    beca = (rng.random(n) < (0.25 - vulnerabilidad * 0.10)).astype(int)
    beca = np.clip(beca, 0, 1)
    
    riesgo_score = (
        -0.05 * tasa_asistencia
        -0.04 * promedio_calificaciones
        +0.55 * reprobaciones_previas
        +0.30 * incidencias_disciplinarias
        +0.40 * cambios_escuela
        +0.04 * distancia_escuela
        +0.9 * trabaja_estudiante
        -0.6 * acceso_internet
        -0.5 * beca
        +1.5 * (nivel_socioeconomico == 'Bajo').astype(int)
        +1.2 * (apoyo_familiar == 'Bajo').astype(int)
        + rng.normal(0, 1.5, n)
    )
    prob_desercion = 1 / (1 + np.exp(-(riesgo_score - np.median(riesgo_score))))
    desercion = (rng.random(n) < prob_desercion * 0.42).astype(int)
    
    df = pd.DataFrame({
        'edad': edad,
        'genero': genero,
        'zona': zona,
        'promedio_calificaciones': np.round(promedio_calificaciones, 1),
        'tasa_asistencia': np.round(tasa_asistencia, 1),
        'reprobaciones_previas': reprobaciones_previas,
        'nivel_socioeconomico': nivel_socioeconomico,
        'educacion_padre': educacion_padre,
        'educacion_madre': educacion_madre,
        'trabaja_estudiante': trabaja_estudiante,
        'distancia_escuela': np.round(distancia_escuela, 1),
        'acceso_internet': acceso_internet,
        'actividades_extracurriculares': actividades_extracurriculares,
        'apoyo_familiar': apoyo_familiar,
        'incidencias_disciplinarias': incidencias_disciplinarias,
        'cambios_escuela': cambios_escuela,
        'beca': beca,
        'desercion': desercion
    })
    
    # Inyectar problemas de calidad controlados
    idx_nulos_padre = rng.choice(n, int(n * 0.032), replace=False)
    df.loc[idx_nulos_padre, 'educacion_padre'] = np.nan
    idx_nulos_dist = rng.choice(n, int(n * 0.018), replace=False)
    df.loc[idx_nulos_dist, 'distancia_escuela'] = np.nan
    idx_dup = rng.choice(n, 27, replace=False)
    df = pd.concat([df, df.loc[idx_dup]], ignore_index=True)
    idx_outlier = rng.choice(len(df), 12, replace=False)
    df.loc[idx_outlier, 'tasa_asistencia'] = rng.uniform(101, 130, 12)
    return df

df = generar_dataset()
print(f'Dataset generado: {df.shape[0]} registros, {df.shape[1]} columnas')
df.head()


### Análisis Exploratorio de Datos (EDA) y Diagnóstico Inicial


In [ ]:
print('=== Valores nulos por columna ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Registros duplicados ===')
print(f'Duplicados: {df.duplicated().sum()}')
print('\n=== Asistencia fuera de rango (>100%) ===')
print(f'Registros: {(df["tasa_asistencia"] > 100).sum()}')


### Distribución de la variable objetivo (Deserción)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
counts = df['desercion'].value_counts()
ax[0].bar(['No deserción (0)', 'Deserción (1)'], counts.values, color=['#2C5F2D', '#B85042'])
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 30, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
ax[0].set_title('Distribución de la Variable Objetivo')
ax[0].set_ylabel('Número de Estudiantes')

counts.plot.pie(autopct='%1.1f%%', labels=['No desertó', 'Desertó'], colors=['#2C5F2D', '#B85042'], ax=ax[1], ylabel='')
ax[1].set_title('Proporción de Deserción')
plt.tight_layout()
plt.show()


## 3️⃣ Fase 3: Preparación de los Datos (Data Preparation)

### Limpieza de Datos
En esta sección corregiremos los duplicados, los valores nulos mediante imputación (mediana para variables numéricas, moda para categóricas) y aplicaremos capping sobre los outliers de asistencia.


In [ ]:
df_clean = df.copy()
# 1. Eliminar duplicados exactos
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
# 2. Capping a la asistencia
df_clean['tasa_asistencia'] = df_clean['tasa_asistencia'].clip(upper=100)
# 3. Imputar valores nulos
moda_padre = df_clean['educacion_padre'].mode()[0]
df_clean['educacion_padre'] = df_clean['educacion_padre'].fillna(moda_padre)
mediana_dist = df_clean['distancia_escuela'].median()
df_clean['distancia_escuela'] = df_clean['distancia_escuela'].fillna(mediana_dist)
print(f'Nulos restantes: {df_clean.isnull().sum().sum()}')
print(f'Dataset depurado: {df_clean.shape[0]} registros')


### Codificación y División de Datos
Codificamos las variables ordinales asignándoles valores numéricos ordenados, codificamos las categóricas nominales en variables dummy (One-Hot) y dividimos los datos de manera estratificada en entrenamiento (85%) y prueba (15%).


In [ ]:
# Mapeos ordinales
ordinal_maps = {
    'nivel_socioeconomico': {'Bajo': 0, 'Medio': 1, 'Alto': 2},
    'educacion_padre': {'Ninguno': 0, 'Primaria': 1, 'Secundaria': 2, 'Superior': 3},
    'educacion_madre': {'Ninguno': 0, 'Primaria': 1, 'Secundaria': 2, 'Superior': 3},
    'apoyo_familiar': {'Bajo': 0, 'Medio': 1, 'Alto': 2},
}
df_proc = df_clean.copy()
for col, mapping in ordinal_maps.items():
    df_proc[col] = df_proc[col].map(mapping)

# Dummy encoding para nominales
df_proc['genero_M'] = (df_proc['genero'] == 'M').astype(int)
df_proc['zona_Urbana'] = (df_proc['zona'] == 'Urbana').astype(int)
df_proc = df_proc.drop(columns=['genero', 'zona'])

# 1. Separar datos para Clasificación (Objetivo: Deserción)
X_clf = df_proc.drop(columns=['desercion'])
y_clf = df_proc['desercion']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.15, stratify=y_clf, random_state=RANDOM_STATE
)

# 2. Separar datos para Regresión (Objetivo: promedio_calificaciones)
X_reg = df_proc.drop(columns=['promedio_calificaciones', 'desercion'])
y_reg = df_proc['promedio_calificaciones']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.15, random_state=RANDOM_STATE
)
print('División y codificación completadas.')


### Escalamiento y Balanceo (SMOTE)
Para garantizar la convergencia del modelo clasificador (Regresión Logística) y evitar sesgos de escala, normalizamos las variables numéricas al rango [0, 1]. Luego aplicamos SMOTE para corregir el desbalance de clases sobre los datos de entrenamiento de clasificación.


In [ ]:
num_vars_clf = ['edad', 'promedio_calificaciones', 'tasa_asistencia', 
                'reprobaciones_previas', 'distancia_escuela', 
                'incidencias_disciplinarias', 'cambios_escuela']
scaler_clf = MinMaxScaler()
X_train_clf_scaled = X_train_clf.copy()
X_test_clf_scaled = X_test_clf.copy()
X_train_clf_scaled[num_vars_clf] = scaler_clf.fit_transform(X_train_clf[num_vars_clf])
X_test_clf_scaled[num_vars_clf] = scaler_clf.transform(X_test_clf[num_vars_clf])

smote = SMOTE(sampling_strategy=0.6667, random_state=RANDOM_STATE)
X_train_res_clf, y_train_res_clf = smote.fit_resample(X_train_clf_scaled, y_train_clf)

num_vars_reg = ['edad', 'tasa_asistencia', 'reprobaciones_previas', 
                'distancia_escuela', 'incidencias_disciplinarias', 'cambios_escuela']
scaler_reg = MinMaxScaler()
X_train_reg_scaled = X_train_reg.copy()
X_test_reg_scaled = X_test_reg.copy()
X_train_reg_scaled[num_vars_reg] = scaler_reg.fit_transform(X_train_reg[num_vars_reg])
X_test_reg_scaled[num_vars_reg] = scaler_reg.transform(X_test_reg[num_vars_reg])

print(f'Muestras entrenamiento Clasificación (SMOTE): {len(X_train_res_clf)}')
print(f'Muestras entrenamiento Regresión: {len(X_train_reg_scaled)}')


## 4️⃣ Fase 4: Modelado (Modeling)

### 4.1 Clasificador: Regresión Logística y Random Forest


In [ ]:
# Regresión Logística (Modelo Base)
model_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
model_lr.fit(X_train_res_clf, y_train_res_clf)

# Random Forest (Mejor modelo clasificador)
model_rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=3, 
    class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1
)
model_rf.fit(X_train_res_clf, y_train_res_clf)
print('Modelos clasificadores entrenados.')


### 4.2 Regresor: Regresión Lineal
Entrenamos el modelo de Regresión Lineal para estimar el promedio de calificaciones del estudiante basándose en su perfil socioeconómico e historial conductual.


In [ ]:
model_lin = LinearRegression()
model_lin.fit(X_train_reg_scaled, y_train_reg)
print('Modelo regresor entrenado.')


## 5️⃣ Fase 5: Evaluación (Evaluation)

### 5.1 Evaluación del Clasificador (Deserción)


In [ ]:
def evaluar_clasificador(modelo, X, y, nombre):
    y_pred = modelo.predict(X)
    y_proba = modelo.predict_proba(X)[:, 1]
    return {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1-Score': f1_score(y, y_pred),
        'AUC-ROC': roc_auc_score(y, y_proba),
        'AP (AUC-PR)': average_precision_score(y, y_proba),
        'Kappa': cohen_kappa_score(y, y_pred)
    }

res_lr = evaluar_clasificador(model_lr, X_test_clf_scaled, y_test_clf, 'Regresión Logística')
res_rf = evaluar_clasificador(model_rf, X_test_clf_scaled, y_test_clf, 'Random Forest')
pd.DataFrame([res_lr, res_rf]).set_index('Modelo').round(3)


### Matriz de Confusión y Curva ROC del Modelo Seleccionado (Random Forest)


In [ ]:
y_pred_rf = model_rf.predict(X_test_clf_scaled)
cm = confusion_matrix(y_test_clf, y_pred_rf)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Deserción', 'Deserción'], yticklabels=['No Deserción', 'Deserción'])
plt.title('Matriz de Confusión - Random Forest (Test)')
plt.ylabel('Realidad')
plt.xlabel('Predicción')
plt.tight_layout()
plt.show()


### 5.2 Evaluación del Regresor (Promedio de Calificaciones)


In [ ]:
y_pred_reg = model_lin.predict(X_test_reg_scaled)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print('=== Evaluación del Modelo Regresor (Regresión Lineal) ===')
print(f'Mean Squared Error (MSE):       {mse:.3f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.3f}')
print(f'R² Score:                       {r2:.3f}')


## 6️⃣ Fase 6: Despliegue y Mantenimiento (Deployment & Maintenance)

### 6.1 Prototipo de Sistema de Alerta Temprana (SAT)
El sistema de alerta temprana toma las probabilidades de deserción y clasifica al estudiante en tres niveles de riesgo accionables.


In [ ]:
def clasificar_riesgo(prob):
    if prob < 0.30: return 'BAJO'
    elif prob < 0.60: return 'MEDIO'
    else: return 'ALTO'

y_proba_test_rf = model_rf.predict_proba(X_test_clf_scaled)[:, 1]
reporte_test = X_test_clf.copy()
reporte_test['probabilidad_desercion_pct'] = (y_proba_test_rf * 100).round(1)
reporte_test['nivel_riesgo'] = [clasificar_riesgo(p) for p in y_proba_test_rf]
print('Distribución de niveles de riesgo en el conjunto de prueba:')
print(reporte_test['nivel_riesgo'].value_counts())


### 6.2 Reflexión Ética y Sesgos Algorítmicos

El análisis desagregado por grupos muestra diferencias:
- **Sesgo por zona:** Los estudiantes de zonas rurales pueden tener tasas ligeramente superiores de falsos positivos debido a las diferencias de vulnerabilidad socioeconómica.
- **Sesgo por género:** El recall del modelo varía levemente entre géneros, indicando patrones de abandono no lineales en hombres respecto a mujeres.

**Recomendaciones para un uso responsable:**
1. **Supervisión humana obligatoria:** Ningún algoritmo debe sustituir al docente u orientador. La alerta debe ser validada.
2. **Privacidad de datos:** Los datos socioeconómicos de los alumnos son sensibles y deben protegerse bajo marcos legales.
3. **Auditorías de concepto (Concept Drift):** El modelo debe re-entrenarse periódicamente con datos institucionales reales.

---
*Cuaderno finalizado — Proyecto Final Integrador, Maestría en IA.*
